In [0]:
# Librerias base de Spark para todo el cuaderno
from pyspark.sql import functions as F
from pyspark.sql import Window
import time

print("Spark:", spark.version)
print("Este cuaderno esta disenado para Databricks con Spark DataFrames, SQL, Volumes y Parquet.")

Spark: 4.1.0
Este cuaderno esta disenado para Databricks con Spark DataFrames, SQL, Volumes y Parquet.


## Comprobar Datalake

In [0]:
# Corre esto para saber qué carpeta está vacía
for carpeta in ["contratos", "adiciones", "ejecucion", "divipola"]:
    try:
        print(f"📁 Contenido de '{carpeta}':")
        archivos = dbutils.fs.ls(f"/Volumes/workspace/default/taller_final/raw/{carpeta}")
        print(f"   ✅ OK. Tiene {len(archivos)} elementos.")
    except Exception as e:
        print(f"   ❌ ERROR EN CARPETA '{carpeta}': {e}")

📁 Contenido de 'contratos':
   ✅ OK. Tiene 33 elementos.
📁 Contenido de 'adiciones':
   ✅ OK. Tiene 219 elementos.
📁 Contenido de 'ejecucion':
   ✅ OK. Tiene 23 elementos.
📁 Contenido de 'divipola':
   ✅ OK. Tiene 2 elementos.


## 🗺️ 6. CRUCE TERRITORIAL MAESTRO - ESTRATEGIA DE EXTRACCIÓN INVERSA CON DIAGNÓSTICO DE VACÍOS

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.window import Window

PATH_BASE = "/Volumes/workspace/default/taller_final/raw"

# ==========================================
# 1. Cargar las bases de datos
# ==========================================
df_contratos = spark.read.parquet(f"{PATH_BASE}/contratos/*")
df_adiciones = spark.read.parquet(f"{PATH_BASE}/adiciones/*")
df_ejecuciones = spark.read.parquet(f"{PATH_BASE}/ejecucion/*")
df_divipola = spark.read.parquet(f"{PATH_BASE}/divipola/*")


# ==========================================
# 2. Conversión de Tipos (Contratos)
# ==========================================
df_contratos_limpio = df_contratos \
    .withColumn("valor_del_contrato", F.col("valor_del_contrato").cast("double")) \
    .withColumn("fecha_de_firma", F.to_date(F.col("fecha_de_firma"))) \
    .withColumn("fecha_de_inicio_del_contrato", F.to_date(F.col("fecha_de_inicio_del_contrato"))) \
    .withColumn("fecha_de_fin_del_contrato", F.to_date(F.col("fecha_de_fin_del_contrato")))


# ==========================================
# 3. Resumir adiciones por contrato
# ==========================================
df_adiciones_resumen = df_adiciones \
    .groupBy("id_contrato") \
    .agg(
        F.count("identificador").alias("total_numero_adiciones")
    )


# ==========================================
# 4. Último avance de ejecución
# ==========================================
ventana_ejecucion = Window.partitionBy("identificadorcontrato").orderBy(F.col("fechacreacion").desc())

df_ejecucion_ultimo = df_ejecuciones \
    .withColumn("rn", F.row_number().over(ventana_ejecucion)) \
    .filter(F.col("rn") == 1) \
    .drop("rn") \
    .select(
        F.col("identificadorcontrato").alias("id_contrato_ejecucion"), 
        "porcentaje_de_avance_real", 
        "fechacreacion"
    )


# ==========================================
# 5. Integración de bases (Contratos + Adiciones + Ejecución)
# ==========================================
df_integrado = df_contratos_limpio.join(
    df_adiciones_resumen, 
    on="id_contrato", 
    how="left"
)

df_integrado = df_integrado.join(
    df_ejecucion_ultimo, 
    df_integrado["id_contrato"] == df_ejecucion_ultimo["id_contrato_ejecucion"], 
    how="left"
).drop("id_contrato_ejecucion")

df_integrado = df_integrado.fillna({
    "total_numero_adiciones": 0, 
    "porcentaje_de_avance_real": 0.0
})


# ==========================================
# 6. Cruce Territorial con DIVIPOLA (Ultra-Optimizado)
# ==========================================
def homologar_territorio(columna):
    c = F.upper(F.col(columna))
    c = F.regexp_replace(c, "[ÁÄÂÀ]", "A")
    c = F.regexp_replace(c, "[ÉËÊÈ]", "E")
    c = F.regexp_replace(c, "[ÍÏÎÌ]", "I")
    c = F.regexp_replace(c, "[ÓÖÔÒ]", "O")
    c = F.regexp_replace(c, "[ÚÜÛÙ]", "U")
    c = F.regexp_replace(c, "[,.]", "")
    
    # Super-Diccionario de los casos más famosos en Colombia
    c = F.when(c.like("%BOGOTA%"), "BOGOTA") \
         .when(c.like("%DISTRITO CAPITAL%"), "BOGOTA") \
         .when(c == "DC", "BOGOTA") \
         .when(c.like("%VALLE DEL CAUCA%"), "VALLE DEL CAUCA") \
         .when(c == "VALLE", "VALLE DEL CAUCA") \
         .when(c.like("%NORTE DE SANTANDER%"), "NORTE DE SANTANDER") \
         .when(c == "NORTE SANTANDER", "NORTE DE SANTANDER") \
         .when(c.like("%SAN ANDRES%PROVIDENCIA%"), "ARCHIPIELAGO DE SAN ANDRES PROVIDENCIA Y SANTA CATALINA") \
         .when(c.like("%ARCHIPIELAGO DE SAN ANDRES%"), "ARCHIPIELAGO DE SAN ANDRES PROVIDENCIA Y SANTA CATALINA") \
         .when(c.like("%CUCUTA%"), "SAN JOSE DE CUCUTA") \
         .when(c.like("%CARTAGENA%"), "CARTAGENA DE INDIAS") \
         .when(c == "BUGA", "GUADALAJARA DE BUGA") \
         .when(c.like("%MOMPO%"), "SANTA CRUZ DE MOMPOX") \
         .when(c.like("%TUMACO%"), "SAN ANDRES DE TUMACO") \
         .when(c == "QUIBDO", "SAN FRANCISCO DE QUIBDO") \
         .when(c.like("%GUAJIRA%"), "LA GUAJIRA") \
         .otherwise(c)
         
    c = F.regexp_replace(c, "\s+", " ")
    return F.trim(c)

df_integrado_limpio = df_integrado \
    .withColumn("ciudad_limpia", homologar_territorio("ciudad")) \
    .withColumn("dpto_limpio", homologar_territorio("departamento"))

df_divipola_limpio = df_divipola \
    .withColumn("nom_mpio_limpio", homologar_territorio("nom_mpio")) \
    .withColumn("dpto_divi_limpio", homologar_territorio("dpto"))

# Ejecutamos el Join
df_final = df_integrado_limpio.join(
    df_divipola_limpio,
    (df_integrado_limpio["ciudad_limpia"] == df_divipola_limpio["nom_mpio_limpio"]) & 
    (df_integrado_limpio["dpto_limpio"] == df_divipola_limpio["dpto_divi_limpio"]),
    how="left"
)

df_final = df_final.drop("ciudad_limpia", "dpto_limpio", "nom_mpio_limpio", "dpto_divi_limpio")


# ==========================================
# 7. Reporte de registros sin cruce territorial
# ==========================================
# Excluimos los "No Definido" y "Colombia" porque son imposibles de cruzar por naturaleza
df_sin_cruce = df_final.filter(
    F.col("cod_mpio").isNull() & 
    ~F.upper(F.col("ciudad")).isin("NO DEFINIDO", "COLOMBIA", "SIN DESCRIPCION") &
    ~F.upper(F.col("departamento")).isin("NO DEFINIDO", "COLOMBIA", "SIN DESCRIPCION")
)

print(f"Total de contratos SIN cruce territorial (Verdaderos Errores): {df_sin_cruce.count()}")

# Agrupamos los ofensores REALES para ver qué nos falta arreglar
df_ofensores = df_sin_cruce.groupBy("departamento", "ciudad") \
    .count() \
    .orderBy(F.col("count").desc())

display(df_ofensores)

<>:103: SyntaxWarning: invalid escape sequence '\s'
<>:103: SyntaxWarning: invalid escape sequence '\s'
/home/spark-897de89b-74b2-47ea-8241-46/.ipykernel/13773/command-5656854416013565-3920858308:103: SyntaxWarning: invalid escape sequence '\s'
  c = F.regexp_replace(c, "\s+", " ")
<unknown>:103: SyntaxWarning: invalid escape sequence '\s'


Total de contratos SIN cruce territorial (Verdaderos Errores): 106232


<unknown>:103: SyntaxWarning: invalid escape sequence '\s'


departamento,ciudad,count
Valle del Cauca,Cali,101575
Cauca,Piendamó,1397
Antioquia,Santafé De Antioquia,950
Antioquia,Don Matías,625
Tolima,Mariquita,502
Antioquia,San Vicente,454
Antioquia,San Pedro,434
Antioquia,San Andrés,178
Putumayo,Leguízamo,82
Cesar,Manaure,13


<unknown>:103: SyntaxWarning: invalid escape sequence '\s'


## Retirar contratos sin cruce o ambiguos en su localización

In [0]:
# ==========================================
# 8. Aislamiento y depuración final (Separado en 3 cubetas)
# ==========================================

# CUBETA 1: GOLD (Cruces perfectos)
df_final_gold = df_final.filter(F.col("cod_mpio").isNotNull())

# CUBETA 2: NACIONALES / NO DEFINIDOS (No cruzaron porque no tienen un municipio específico)
df_nacionales = df_final.filter(
    F.col("cod_mpio").isNull() & 
    (F.upper(F.col("ciudad")).isin("NO DEFINIDO", "COLOMBIA", "SIN DESCRIPCION") |
     F.upper(F.col("departamento")).isin("NO DEFINIDO", "COLOMBIA", "SIN DESCRIPCION"))
)

# CUBETA 3: ERRORES DE DIGITACIÓN (Los verdaderos 106k ofensores)
df_errores = df_final.filter(
    F.col("cod_mpio").isNull() & 
    ~F.upper(F.col("ciudad")).isin("NO DEFINIDO", "COLOMBIA", "SIN DESCRIPCION") &
    ~F.upper(F.col("departamento")).isin("NO DEFINIDO", "COLOMBIA", "SIN DESCRIPCION")
)

print(f"✅ Cubeta 1 (Analítica Gold)        : {df_final_gold.count()}")
print(f"🌍 Cubeta 2 (Nacionales/Sin Ciudad): {df_nacionales.count()}")
print(f"❌ Cubeta 3 (Errores de SECOP)     : {df_errores.count()}")

✅ Cubeta 1 (Analítica Gold)        : 1173516
🌍 Cubeta 2 (Nacionales/Sin Ciudad): 296227
❌ Cubeta 3 (Errores de SECOP)     : 106232


# 🧠 ACTIVIDAD 3: PROCESAMIENTO DE TEXTO NO ESTRUCTURADO Y DETECCIÓN DE TEMAS

## Explicacion reglas para filtrar y agrupar tematicas

13 reglas y palabras clave (Expresiones Regulares / Regex) que diseñamos para clasificar los contratos.

Estas reglas se aplican en orden jerárquico (de la más específica a la más general) sobre una columna de texto unificada y limpia (sin tildes ni mayúsculas):

🏥 1. Reglas Ultra-Específicas (Prioridad Alta)
Atrapan los contratos de mayor impacto social o de nicho, evitando que caigan en categorías genéricas.

Salud y Bienestar: salud | hospital | medicament | clinic | medico | enfermer | paciente | covid | vacuna | odontolog | psicolog
Educación y Primera Infancia: educacion | colegio | escuela | universidad | docent | estudiant | pedagog | sena | icbf
Alimentación (PAE / Mercados): alimentacion | comida | restaurante | pae | alimento | racion | mercado
Infraestructura y Obras: construccion | obras | mantenimiento vial | carretera | puente | pavimentacion | adecuacion | remodelacion | ingenieria civil
Tecnología e Informática: software | hardware | computador | tecnolog | sistema | informatic | licencia | nube | internet | conectividad
🔎 2. Reglas Misionales Transversales (Prioridad Media)
Atrapan contratos de soporte institucional, logística y bienes públicos.

Interventoría y Auditoría: interventoria | auditoria | revisoria | fiscalizac
Seguros y Pólizas: seguro | poliza | amparo | siniestro | todo riesgo
Seguridad y Vigilancia: vigilancia | seguridad privada | guarda | escolta | monitoreo | alarma | cctv
Transporte y Vehículos: transporte | vehiculo | camion | combustible | gasolina | acpm | soat | llantas | automotor | tiquete | pasaje
Medio Ambiente y Servicios Públicos: ambiental | arbol | residuos | basura | reciclaje | acueducto | alcantarillado | agua potable | reforestacion
Cultura, Deporte y Eventos: deporte | cultura | artist | entrenador | cancha | torneo | musica | evento | logistica | tarima | sonido | festival | navideñ
💼 3. Reglas de Funcionamiento Interno (Prioridad Baja - "Red de Seguridad")
Diseñadas para atrapar el volumen masivo de contratos del día a día (funcionamiento del Estado) que no son proyectos de inversión específicos.

Suministros, Dotación y Aseo: suministro | papeleria | dotacion | uniforme | escritorio | papel | impresora | tinta | aseo | cafeteria | materiales
Servicios Profesionales / Apoyo a la Gestión: apoyo | gestion | profesional | prestacion de servicios | honorarios | abogado | juridic | contable | asesor | administrativo
🗑️ 4. Categoría por Defecto
Servicios Generales / Otro: Cualquier contrato cuyo objeto o descripción no contenga ninguna de las 80+ palabras clave anteriores.

In [0]:
import pyspark.sql.functions as F

# ==========================================
# 1. Crear 'texto_busqueda' y limpiar tildes
# ==========================================
# Unimos objeto y descripción en una sola columna en minúsculas
df_texto = df_final_gold.withColumn(
    "texto_busqueda",
    F.lower(F.concat_ws(" ", F.col("objeto_del_contrato"), F.col("descripcion_del_proceso")))
)

# Limpiamos tildes para evitar falsos negativos
df_texto = df_texto.withColumn(
    "texto_busqueda", F.regexp_replace("texto_busqueda", "[áäâà]", "a")
).withColumn(
    "texto_busqueda", F.regexp_replace("texto_busqueda", "[éëêè]", "e")
).withColumn(
    "texto_busqueda", F.regexp_replace("texto_busqueda", "[íïîì]", "i")
).withColumn(
    "texto_busqueda", F.regexp_replace("texto_busqueda", "[óöôò]", "o")
).withColumn(
    "texto_busqueda", F.regexp_replace("texto_busqueda", "[úüûù]", "u")
)


# ==========================================
# 2. Crear 'temas_detectados' mediante reglas (Regex Masivo)
# ==========================================
# Reglas ultra-específicas
regla_salud = F.col("texto_busqueda").rlike("salud|hospital|medicament|clinic|medico|enfermer|paciente|covid|vacuna|odontolog|psicolog")
regla_educacion = F.col("texto_busqueda").rlike("educacion|colegio|escuela|universidad|docent|estudiant|pedagog|sena|icbf")
regla_pae = F.col("texto_busqueda").rlike("alimentacion|comida|restaurante|pae|alimento|racion|mercado")
regla_infra = F.col("texto_busqueda").rlike("construccion|obras|mantenimiento vial|carretera|puente|pavimentacion|adecuacion|remodelacion|ingenieria civil")
regla_tecnologia = F.col("texto_busqueda").rlike("software|hardware|computador|tecnolog|sistema|informatic|licencia|nube|internet|conectividad")

# Reglas misionales transversales
regla_interventoria = F.col("texto_busqueda").rlike("interventoria|auditoria|revisoria|fiscalizac")
regla_seguros = F.col("texto_busqueda").rlike("seguro|poliza|amparo|siniestro|todo riesgo")
regla_seguridad = F.col("texto_busqueda").rlike("vigilancia|seguridad privada|guarda|escolta|monitoreo|alarma|cctv")
regla_transporte = F.col("texto_busqueda").rlike("transporte|vehiculo|camion|combustible|gasolina|acpm|soat|llantas|automotor|tiquete|pasaje")
regla_ambiental = F.col("texto_busqueda").rlike("ambiental|arbol|residuos|basura|reciclaje|acueducto|alcantarillado|agua potable|reforestacion")
regla_cultura_eventos = F.col("texto_busqueda").rlike("deporte|cultura|artist|entrenador|cancha|torneo|musica|evento|logistica|tarima|sonido|festival|navideñ")

# Reglas de funcionamiento interno
regla_suministros = F.col("texto_busqueda").rlike("suministro|papeleria|dotacion|uniforme|escritorio|papel|impresora|tinta|aseo|cafeteria|materiales")
regla_apoyo_gestion = F.col("texto_busqueda").rlike("apoyo|gestion|profesional|prestacion de servicios|honorarios|abogado|juridic|contable|asesor|administrativo")


# Aplicamos las reglas de arriba hacia abajo (jerarquía)
df_temas = df_texto.withColumn(
    "temas_detectados",
    F.when(regla_salud, "Salud y Bienestar")
     .when(regla_educacion, "Educación y Primera Infancia")
     .when(regla_pae, "Alimentación (PAE / Mercados)")
     .when(regla_infra, "Infraestructura y Obras")
     .when(regla_tecnologia, "Tecnología e Informática")
     .when(regla_interventoria, "Interventoría y Auditoría")
     .when(regla_seguros, "Seguros y Pólizas")
     .when(regla_seguridad, "Seguridad y Vigilancia")
     .when(regla_transporte, "Transporte y Vehículos")
     .when(regla_ambiental, "Medio Ambiente y Servicios Públicos")
     .when(regla_cultura_eventos, "Cultura, Deporte y Eventos")
     .when(regla_suministros, "Suministros, Dotación y Aseo")
     .when(regla_apoyo_gestion, "Servicios Profesionales / Apoyo a la Gestión")
     .otherwise("Servicios Generales / Otro") 
)


# ==========================================
# 3. Resumen de contratos por tema
# ==========================================
resumen_temas = df_temas.groupBy("temas_detectados").agg(
    F.count("id_contrato").alias("cantidad_contratos"),
    F.sum("valor_del_contrato").alias("valor_total_involucrado")
).orderBy(F.col("cantidad_contratos").desc())

# Mostrar la tabla de resumen
print("=== RESUMEN POR TEMA ===")
display(resumen_temas)

# Mostrar una muestra de contratos individuales y cómo fueron clasificados
print("=== MUESTRA DE CONTRATOS Y SU CLASIFICACIÓN ===")
display(df_temas.select("id_contrato", "objeto_del_contrato", "temas_detectados").limit(10))

=== RESUMEN POR TEMA ===


temas_detectados,cantidad_contratos,valor_total_involucrado
Servicios Profesionales / Apoyo a la Gestión,405394,2.7374080052608E13
Salud y Bienestar,206306,1.5596785379783E13
Alimentación (PAE / Mercados),145620,3.3555713261286E13
Educación y Primera Infancia,109836,2.0306829775324E13
Tecnología e Informática,89583,1.4180826143268E13
"Cultura, Deporte y Eventos",62211,5.602444567131E12
Servicios Generales / Otro,41081,1.7559776620703E13
Infraestructura y Obras,28135,2.379487551278E13
Transporte y Vehículos,21420,3.205053649247E12
Medio Ambiente y Servicios Públicos,21310,2.234324741063E12


=== MUESTRA DE CONTRATOS Y SU CLASIFICACIÓN ===


id_contrato,objeto_del_contrato,temas_detectados
CO1.PCCNTR.9046436,Brindar apoyo en el desarrollo de estrategias de carácter administrativo para el trámite de los procesos a cargo de la DEE y aquellos relacionados con los fondos de energía eléctrica.,Servicios Profesionales / Apoyo a la Gestión
CO1.PCCNTR.9059978,CONTRATO DE PRESTACIÓN DE SERVICIOS DE UN PROFESIONAL EN AREAS DE LA SALUD PARA APOYAR EN LA EJCUCIÓN DEL PAMEC (PROGRAMA DE AUDITORIA PARA EL MEJORAMIENTO DE LA CALIDAD) DE LA OFICINA DE PRESTACIÓN Y DESARROLLO DE SERVICIOS DE LA SALUD DE LA SECRETARIA DE SALUD DEPARTAMENTAL DEL PUTUMAYO,Salud y Bienestar
CO1.PCCNTR.9042607,Prestar Servicios Apoyo Pedagogico En Las Unidades De Servicio De Atencion Directa A La Primera Infancia Que Le Sean Asignadas Por La Direccion Regional Para Realizar La Atencion Directa A Las Ninias Y Ninios Vinculados En Los Servicios De Educacion Inicial Conforme A Los Lineamientos Manuales Protocolos Y Guias Vigentes Aplicables A La Modalidad O Servicio Correspondiente.,Educación y Primera Infancia
CO1.PCCNTR.9022280,5_9504_067 Prestar los servicios personales de carácter temporal como instructor para impartir formación titulada y/o complementaria; presencial y/o virtual; en los programas de formación financiados con recursos regulares; en el Complejo Tecnológico Agroindustrial Pecuario y Turístico.,Tecnología e Informática
CO1.PCCNTR.9001987,PRESTACIÓN DE SERVICIOS DE APOYO A LA GESTIÓN PARA LA ADECUACIÓN Y MANTENIMIENTO A LAS REDES ELÉCTRICAS Y DE ALUMBRADO PÚBLICO EN EL MUNICIPIO DE SAN AGUSTÍN; HUILA,Infraestructura y Obras
CO1.PCCNTR.9075021,Prestación de servicios profesionales para el fortalecimiento del ordenamiento territorial y desarrollo urbano y rural,Servicios Profesionales / Apoyo a la Gestión
CO1.PCCNTR.9017962,PRESTACION DE SERVICIOS TECNICOS PARA EL FORTALECIMIENTO INSTITUCIONAL A LA OFICINA DE CONTROL INTERNO DEL MUNICIPIO DE MAGANGUE.,Servicios Profesionales / Apoyo a la Gestión
CO1.PCCNTR.9063995,PRESTACIÓN DE SERVICIOS PROFESIONALES PARA REALIZAR CUBRIMIENTO EN LA SECRETARIA DE COMUNICACIONES DE LA GOBERNACIÓN DE SUCRE,Servicios Profesionales / Apoyo a la Gestión
CO1.PCCNTR.9042382,BRINDAR APOYO TÉCNICO A LA OFICINA DE OBRAS CIVILES EN LOS PROCESOS DE MODERNIZACIÓN; MANTENIMIENTO Y AMPLIACIÓN DEL SISTEMA DE REDES ELÉCTRICAS DE ALUMBRADO PÚBLICO DEL MUNICIPIO DE LA PLATA; HUILA.,Infraestructura y Obras
CO1.PCCNTR.9052935,PRESTAR SERVICIOS PROFESIONALES PARA ATENDER ASUNTOS DE CARÁCTER TÉCNICO DE LOS CONTRATOS Y/O CONVENIOS DEL INVIAS EN LA SUBDIRECCIÓN DE VÍAS REGIONALES EN EL MARCO DEL PROGRAMA CAMINOS COMUNITARIOS DE LA PAZ TOTAL,Servicios Profesionales / Apoyo a la Gestión


## **Metodología y Explicación de la Fórmula del Índice de Prioridad**

Para evaluar el nivel de riesgo o prioridad de auditoría de cada contrato, se diseñó un Índice Heurístico Aditivo en una escala de 0 a 100 puntos. A mayor puntaje, mayor es la probabilidad de que el contrato requiera revisión por posibles anomalías o alta criticidad.

La fórmula matemática implementada es la sumatoria de 6 factores de riesgo: Índice de Prioridad = Puntos(Valor) + Puntos(Adiciones) + Puntos(Avance) + Puntos(Modalidad) + Puntos(Tema) + Puntos(Calidad de Texto)

A continuación, se detalla la asignación de pesos y la justificación técnica de cada variable:

1. Valor del Contrato (Máximo 20 Puntos)
≥ 1.000 Millones (20 pts): Contratos de mega-cuantía. El impacto financiero para el Estado es altísimo, por lo que son prioridad máxima.
≥ 200 Millones (10 pts): Contratos de mediana-alta cuantía.
Menor a 200 Millones (0 pts).
2. Modificaciones y Adiciones (Máximo 20 Puntos)
3 o más adiciones (20 pts): Recurrir a múltiples adiciones (en tiempo o dinero) es un indicador clásico de mala planeación contractual o posibles sobrecostos.
2 adiciones (10 pts).
1 adición (5 pts).
Cero adiciones (0 pts).
3. Avance de Ejecución Bajo (Máximo 20 Puntos)
Avance Menor al 20% (20 pts): Contratos que reportan una ejecución casi nula o estancada levantan una bandera roja operativa inmediata.
Avance ≤ 50% (10 pts): Ejecución a la mitad, requiere monitoreo.
Avance > 50% (0 pts).
4. Modalidad Contractual (Máximo 15 Puntos)
Contratación Directa (15 pts): Las adjudicaciones "a dedo" restringen la pluralidad de oferentes y la libre competencia, por lo que suponen un riesgo de transparencia mucho mayor frente a una Licitación Pública.
Régimen Especial (10 pts): Modalidades con reglas excepcionales.
Otras modalidades abiertas (0 pts).
5. Tema Detectado (Máximo 10 Puntos)
Sectores Críticos (10 pts): Contratos clasificados en "Infraestructura y Obras", "Alimentación (PAE / Mercados)" o "Salud y Bienestar". Históricamente, en Colombia, estos tres sectores concentran los mayores índices de corrupción o interés nacional.
Otros temas (0 pts).
6. Texto Insuficiente u Opaco (Máximo 15 Puntos)
Texto < 50 caracteres (15 pts): Los contratos cuya descripción y objeto sumados tienen menos de 50 letras vulneran el principio de transparencia de los datos abiertos, ocultando la verdadera naturaleza de la compra.
Texto ≥ 50 caracteres (0 pts).
Clasificación Final (Niveles de Prioridad)
Con base en el puntaje total (0 a 100), se establecieron tres niveles de clasificación para optimizar el trabajo de auditoría:

🔴 Alta (Crítico) [≥ 60 Puntos]: Contratos con alertas múltiples (Ej. Multimillonarios, contratados directamente, con varias adiciones y en sectores sensibles). Acción: Auditoría inmediata.
🟡 Media (Monitoreo) [30 - 59 Puntos]: Contratos con 1 o 2 banderas rojas. Acción: Revisión de control preventivo.
🟢 Baja (Rutinario) [< 30 Puntos]: Ejecución normal, bajas cuantías o licitaciones estándar. Acción: Monitoreo automatizado estándar.

In [0]:
import pyspark.sql.functions as F

# ==========================================
# 1. Asignación de Puntajes (Máximo 100 puntos)
# ==========================================
df_prioridad = df_temas.withColumn(
    # A. Valor Alto (Max 20 pts)
    "pts_valor",
    F.when(F.col("valor_del_contrato") >= 1000000000, 20)  
     .when(F.col("valor_del_contrato") >= 200000000, 10)   
     .otherwise(0)
).withColumn(
    # B. Número de Adiciones (Max 20 pts)
    "pts_adiciones",
    F.when(F.col("total_numero_adiciones") >= 3, 20)       
     .when(F.col("total_numero_adiciones") == 2, 10)
     .when(F.col("total_numero_adiciones") == 1, 5)
     .otherwise(0)
).withColumn(
    # C. Avance Bajo (Max 20 pts) - CORREGIDO
    "pts_avance",
    F.when(F.col("porcentaje_de_avance_real").cast("double") < 20.0, 20)    
     .when(F.col("porcentaje_de_avance_real").cast("double") <= 50.0, 10)
     .otherwise(0)
).withColumn(
    # D. Modalidad Contractual (Max 15 pts)
    "pts_modalidad",
    F.when(F.upper(F.col("modalidad_de_contratacion")).like("%DIRECTA%"), 15)
     .when(F.upper(F.col("modalidad_de_contratacion")).like("%ESPECIAL%"), 10)
     .otherwise(0)
).withColumn(
    # E. Tema Detectado (Max 10 pts)
    "pts_tema",
    F.when(F.col("temas_detectados").isin("Infraestructura y Obras", "Alimentación (PAE / Mercados)", "Salud y Bienestar"), 10)
     .otherwise(0)
).withColumn(
    # F. Texto Insuficiente u Opaco (Max 15 pts)
    "pts_texto",
    F.when(F.length(F.col("texto_busqueda")) < 50, 15)     
     .otherwise(0)
)

# ==========================================
# 2. Cálculo del Índice Total
# ==========================================
df_prioridad = df_prioridad.withColumn(
    "indice_prioridad",
    F.col("pts_valor") + F.col("pts_adiciones") + F.col("pts_avance") + 
    F.col("pts_modalidad") + F.col("pts_tema") + F.col("pts_texto")
)

# ==========================================
# 3. Creación de Niveles (Baja, Media, Alta)
# ==========================================
df_prioridad = df_prioridad.withColumn(
    "nivel_prioridad",
    F.when(F.col("indice_prioridad") >= 60, "Alta (Crítico)")
     .when(F.col("indice_prioridad") >= 30, "Media (Monitoreo)")
     .otherwise("Baja (Rutinario)")
)

# Limpiamos las columnas de puntajes temporales para no saturar la tabla final
columnas_temporales = ["pts_valor", "pts_adiciones", "pts_avance", "pts_modalidad", "pts_tema", "pts_texto"]
df_actividad_4_final = df_prioridad.drop(*columnas_temporales)

# ==========================================
# 4. Ranking de Contratos
# ==========================================
ranking_contratos = df_actividad_4_final.select(
    "id_contrato", "nombre_entidad", "valor_del_contrato", 
    "temas_detectados", "indice_prioridad", "nivel_prioridad"
).orderBy(F.col("indice_prioridad").desc(), F.col("valor_del_contrato").desc())

print("=== RANKING TOP 15 CONTRATOS PRIORITARIOS ===")
display(ranking_contratos.limit(15))

# Resumen estadístico de los niveles
print("=== RESUMEN POR NIVEL DE PRIORIDAD ===")
display(df_actividad_4_final.groupBy("nivel_prioridad").count())

=== RANKING TOP 15 CONTRATOS PRIORITARIOS ===


id_contrato,nombre_entidad,valor_del_contrato,temas_detectados,indice_prioridad,nivel_prioridad
CO1.PCCNTR.9281407,MUNICIPIO DE SOLEDAD,2.21663E9,Servicios Generales / Otro,90,Alta (Crítico)
CO1.PCCNTR.8750509,ALCALDÍA DISTRITAL DE SANTA MARTA,8.93052E11,Infraestructura y Obras,85,Alta (Crítico)
CO1.PCCNTR.7455669,GOBERNACIÓN DE RISARALDA**,5.40775229979E11,Salud y Bienestar,85,Alta (Crítico)
CO1.PCCNTR.7969114,DEPARTAMENTO DE ANTIOQUIA//,4.90363335608E11,Infraestructura y Obras,85,Alta (Crítico)
CO1.PCCNTR.9281738,DISTRITO ESPECIAL INDUSTRIAL Y PORTUARIO DE BARRANQUILLA,3.44398E11,Alimentación (PAE / Mercados),85,Alta (Crítico)
CO1.PCCNTR.7862082,DEPARTAMENTO DE ANTIOQUIA//,3.33262100768E11,Alimentación (PAE / Mercados),85,Alta (Crítico)
CO1.PCCNTR.8364608,INSTITUTO DE DEPORTES Y RECREACION DE MEDELLIN,2.20686630166E11,Infraestructura y Obras,85,Alta (Crítico)
CO1.PCCNTR.7870102,MINISTERIO DE AGRICULTURA Y DESARROLLO RURAL,2.05877922523E11,Alimentación (PAE / Mercados),85,Alta (Crítico)
CO1.PCCNTR.9124860,RNEC,1.9391839325E11,Alimentación (PAE / Mercados),85,Alta (Crítico)
CO1.PCCNTR.7377888,RNEC,1.50586014037E11,Alimentación (PAE / Mercados),85,Alta (Crítico)


=== RESUMEN POR NIVEL DE PRIORIDAD ===


nivel_prioridad,count
Baja (Rutinario),62194
Media (Monitoreo),986211
Alta (Crítico),125111


# 🍃 ACTIVIDAD 5: MODELAMIENTO DOCUMENTAL NOSQL (MIGRACIÓN INTEGRAL A MONGO)

In [0]:
import pyspark.sql.functions as F
from datetime import datetime  # <-- Agregamos esto que faltaba

PATH_NOSQL = "/Volumes/workspace/default/taller_final/raw/mongodb_collections"

# 1. Filtramos y creamos la estructura (SOLO PRIORIDADES ALTA Y MEDIA)
df_contratos_operativos_muestra = df_actividad_4_final.filter(
    F.col("nivel_prioridad").isin("Alta (Crítico)")
).select(
    "id_contrato",
    F.struct("nombre_entidad", "nit_entidad", "departamento", "ciudad","longitud","latitud").alias("entidad"),
    F.struct("proveedor_adjudicado", "documento_proveedor").alias("proveedor"),
    F.struct("valor_del_contrato", "total_numero_adiciones", "porcentaje_de_avance_real").alias("operacion"),
    F.struct("temas_detectados", "indice_prioridad", "nivel_prioridad").alias("auditoria"),
    "fecha_de_firma",
    "fecha_de_fin_del_contrato"
)
total_muestra = df_contratos_operativos_muestra.count()
print(f"El nuevo tamaño de la colección (Alta) será de solo: {total_muestra:,} contratos.")
# 2. Sobrescribimos el archivo JSON anterior
df_contratos_operativos_muestra.write.mode("overwrite").json(f"{PATH_NOSQL}/contratos_operativos")
print("¡Colección súper ligera guardada en disco!")


# ---------------------------------------------------------
# Colección 2: alertas_revision
# Colección dedicada solo a los contratos de prioridad "Alta"
# ---------------------------------------------------------
df_alertas_revision = df_actividad_4_final.filter(
    F.col("nivel_prioridad") == "Alta (Crítico)"
).select(
    "id_contrato",
    "nombre_entidad",
    "valor_del_contrato",
    "temas_detectados",
    "indice_prioridad",
    F.current_timestamp().alias("fecha_generacion_alerta"),
    F.lit("Requiere Auditoría Inmediata").alias("estado_alerta")
)


# ---------------------------------------------------------
# Colección 3: entidades_resumen
# Agrupación por entidad estatal
# ---------------------------------------------------------
df_entidades_resumen = df_actividad_4_final.groupBy("nit_entidad", "nombre_entidad").agg(
    F.count("id_contrato").alias("total_contratos_emitidos"),
    F.sum("valor_del_contrato").alias("presupuesto_total_comprometido"),
    F.avg("indice_prioridad").alias("riesgo_promedio_entidad")
)


# ---------------------------------------------------------
# Colección 4: proveedores_resumen
# Agrupación por contratista/proveedor
# ---------------------------------------------------------
df_proveedores_resumen = df_actividad_4_final.groupBy("documento_proveedor", "proveedor_adjudicado").agg(
    F.count("id_contrato").alias("total_contratos_ganados"),
    F.sum("valor_del_contrato").alias("valor_total_adjudicado"),
    F.avg("porcentaje_de_avance_real").alias("avance_promedio_historico")
)


# ---------------------------------------------------------
# Colección 5: temas_resumen
# Agrupación por las categorías que creamos en la Actividad 3
# ---------------------------------------------------------
df_temas_resumen = df_actividad_4_final.groupBy("temas_detectados").agg(
    F.count("id_contrato").alias("volumen_contratos"),
    F.sum("valor_del_contrato").alias("inversion_total_tema"),
    # Contamos cuántas alertas críticas tiene cada sector
    F.sum(F.when(F.col("nivel_prioridad") == "Alta (Crítico)", 1).otherwise(0)).alias("total_alertas_criticas")
)


# ---------------------------------------------------------
# Colección 6: metadata_pipeline
# Documento único de control con la metadata de la ejecución
# ---------------------------------------------------------
total_registros = df_actividad_4_final.count()
fecha_ejecucion = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

# Creamos un DataFrame de una sola fila directamente
df_metadata_pipeline = spark.createDataFrame([{
    "pipeline_id": "SECOP_DATALAKE_001",
    "nombre_proyecto": "Taller Final - Analítica SECOP",
    "fecha_ultima_ejecucion": fecha_ejecucion,
    "total_contratos_procesados": total_registros,
    "capas_procesadas": ["Raw", "Bronze", "Silver", "Gold"],
    "estado_pipeline": "EXITOSO"
}])

print("¡Las 6 Colecciones NoSQL han sido generadas en memoria!")

# Veamos un ejemplo de la colección operativa usando la variable correcta
display(df_contratos_operativos_muestra.limit(5))

El nuevo tamaño de la colección (Alta) será de solo: 125,111 contratos.
¡Colección súper ligera guardada en disco!
¡Las 6 Colecciones NoSQL han sido generadas en memoria!


id_contrato,entidad,proveedor,operacion,auditoria,fecha_de_firma,fecha_de_fin_del_contrato
CO1.PCCNTR.9059978,"List(GOBERNACIÓN DEPARTAMENTO DEL PUTUMAYO, 800094164, Putumayo, Mocoa, -76,654238, 1,151172)","List(DANIEL ANTONIO CUARAN MUCHAVISOY, 18123529)","List(2.509192E7, 35, 0.0)","List(Salud y Bienestar, 65, Alta (Crítico))",2026-01-22,2026-07-10
CO1.PCCNTR.9042382,"List(MUNICIPIO DE LA PLATA, 891180155, Huila, La Plata, -75,891254, 2,389263)","List(ALBERT ANTONIO CASTAÑEDA TRUJILLO, 83116985)","List(1.86E7, 12, 0.0)","List(Infraestructura y Obras, 65, Alta (Crítico))",2026-01-22,2026-07-22
CO1.PCCNTR.9059120,"List(CORPORACION AUTONOMA REGIONAL DE SANTANDER+, 804000292, Santander, San Gil, -73,134776, 6,551952)","List(HAROL YESID SOPO GOMEZ, 1100966768)","List(1.4926482E7, 28, 0.0)","List(Alimentación (PAE / Mercados), 65, Alta (Crítico))",2026-01-22,2026-06-09
CO1.PCCNTR.9034727,"List(CONCEJO MUNICIPAL DE SAN JOSE DE CÚCUTA., 890505717, Norte de Santander, Cúcuta, -72,508178, 7,905725)","List(DAVIZON DAMIAN DIAZ DIAZ, 1007283087)","List(1.4E7, 11, 0.0)","List(Alimentación (PAE / Mercados), 65, Alta (Crítico))",2026-01-22,2026-06-21
CO1.PCCNTR.8974895,"List(MINISTERIO DE MINAS Y ENERGIA, 899999022, Distrito Capital de Bogotá, Bogotá, -74,106992, 4,649251)","List(Leyda Lorena Parra Diaz, 1116546985)","List(1.38E8, 19, 0.0)","List(Alimentación (PAE / Mercados), 65, Alta (Crítico))",2026-01-22,2026-12-31


## Guardar colecciones en el volumen

In [0]:
# Definimos la ruta exacta que solicitaste
PATH_NOSQL = "/Volumes/workspace/default/taller_final/raw/mongodb_collections"

print(f"Guardando colecciones en formato JSON (MongoDB) en: {PATH_NOSQL} ...")

# Guardamos cada DataFrame físico en el disco
df_contratos_operativos_muestra.write.mode("overwrite").json(f"{PATH_NOSQL}/contratos_operativos")
df_alertas_revision.write.mode("overwrite").json(f"{PATH_NOSQL}/alertas_revision")
df_entidades_resumen.write.mode("overwrite").json(f"{PATH_NOSQL}/entidades_resumen")
df_proveedores_resumen.write.mode("overwrite").json(f"{PATH_NOSQL}/proveedores_resumen")
df_temas_resumen.write.mode("overwrite").json(f"{PATH_NOSQL}/temas_resumen")
df_metadata_pipeline.write.mode("overwrite").json(f"{PATH_NOSQL}/metadata_pipeline")

print("¡Listo! Las 6 colecciones están guardadas exitosamente.")

Guardando colecciones en formato JSON (MongoDB) en: /Volumes/workspace/default/taller_final/raw/mongodb_collections ...
¡Listo! Las 6 colecciones están guardadas exitosamente.


In [0]:
dbutils.library.restartPython()

In [0]:
%pip install pymongo

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
%pip install certifi

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:

import pyspark.sql.functions as F
from pyspark.sql.window import Window

PATH_BASE = "/Volumes/workspace/default/taller_final/raw"

# ==========================================
# 1. Cargar las bases de datos
# ==========================================
df_contratos = spark.read.parquet(f"{PATH_BASE}/contratos/*")
df_adiciones = spark.read.parquet(f"{PATH_BASE}/adiciones/*")
df_ejecuciones = spark.read.parquet(f"{PATH_BASE}/ejecucion/*")
df_divipola = spark.read.parquet(f"{PATH_BASE}/divipola/*")


# ==========================================
# 2. Conversión de Tipos (Contratos)
# ==========================================
df_contratos_limpio = df_contratos \
    .withColumn("valor_del_contrato", F.col("valor_del_contrato").cast("double")) \
    .withColumn("fecha_de_firma", F.to_date(F.col("fecha_de_firma"))) \
    .withColumn("fecha_de_inicio_del_contrato", F.to_date(F.col("fecha_de_inicio_del_contrato"))) \
    .withColumn("fecha_de_fin_del_contrato", F.to_date(F.col("fecha_de_fin_del_contrato")))


# ==========================================
# 3. Resumir adiciones por contrato
# ==========================================
df_adiciones_resumen = df_adiciones \
    .groupBy("id_contrato") \
    .agg(
        F.count("identificador").alias("total_numero_adiciones")
    )


# ==========================================
# 4. Último avance de ejecución
# ==========================================
ventana_ejecucion = Window.partitionBy("identificadorcontrato").orderBy(F.col("fechacreacion").desc())

df_ejecucion_ultimo = df_ejecuciones \
    .withColumn("rn", F.row_number().over(ventana_ejecucion)) \
    .filter(F.col("rn") == 1) \
    .drop("rn") \
    .select(
        F.col("identificadorcontrato").alias("id_contrato_ejecucion"), 
        "porcentaje_de_avance_real", 
        "fechacreacion"
    )


# ==========================================
# 5. Integración de bases (Contratos + Adiciones + Ejecución)
# ==========================================
df_integrado = df_contratos_limpio.join(
    df_adiciones_resumen, 
    on="id_contrato", 
    how="left"
)

df_integrado = df_integrado.join(
    df_ejecucion_ultimo, 
    df_integrado["id_contrato"] == df_ejecucion_ultimo["id_contrato_ejecucion"], 
    how="left"
).drop("id_contrato_ejecucion")

df_integrado = df_integrado.fillna({
    "total_numero_adiciones": 0, 
    "porcentaje_de_avance_real": 0.0
})


# ==========================================
# 6. Cruce Territorial con DIVIPOLA (Ultra-Optimizado)
# ==========================================
def homologar_territorio(columna):
    c = F.upper(F.col(columna))
    c = F.regexp_replace(c, "[ÁÄÂÀ]", "A")
    c = F.regexp_replace(c, "[ÉËÊÈ]", "E")
    c = F.regexp_replace(c, "[ÍÏÎÌ]", "I")
    c = F.regexp_replace(c, "[ÓÖÔÒ]", "O")
    c = F.regexp_replace(c, "[ÚÜÛÙ]", "U")
    c = F.regexp_replace(c, "[,.]", "")
    
    # Super-Diccionario de los casos más famosos en Colombia
    c = F.when(c.like("%BOGOTA%"), "BOGOTA") \
         .when(c.like("%DISTRITO CAPITAL%"), "BOGOTA") \
         .when(c == "DC", "BOGOTA") \
         .when(c.like("%VALLE DEL CAUCA%"), "VALLE DEL CAUCA") \
         .when(c == "VALLE", "VALLE DEL CAUCA") \
         .when(c.like("%NORTE DE SANTANDER%"), "NORTE DE SANTANDER") \
         .when(c == "NORTE SANTANDER", "NORTE DE SANTANDER") \
         .when(c.like("%SAN ANDRES%PROVIDENCIA%"), "ARCHIPIELAGO DE SAN ANDRES PROVIDENCIA Y SANTA CATALINA") \
         .when(c.like("%ARCHIPIELAGO DE SAN ANDRES%"), "ARCHIPIELAGO DE SAN ANDRES PROVIDENCIA Y SANTA CATALINA") \
         .when(c.like("%CUCUTA%"), "SAN JOSE DE CUCUTA") \
         .when(c.like("%CARTAGENA%"), "CARTAGENA DE INDIAS") \
         .when(c == "BUGA", "GUADALAJARA DE BUGA") \
         .when(c.like("%MOMPO%"), "SANTA CRUZ DE MOMPOX") \
         .when(c.like("%TUMACO%"), "SAN ANDRES DE TUMACO") \
         .when(c == "QUIBDO", "SAN FRANCISCO DE QUIBDO") \
         .when(c.like("%GUAJIRA%"), "LA GUAJIRA") \
         .otherwise(c)
         
    c = F.regexp_replace(c, "\s+", " ")
    return F.trim(c)

df_integrado_limpio = df_integrado \
    .withColumn("ciudad_limpia", homologar_territorio("ciudad")) \
    .withColumn("dpto_limpio", homologar_territorio("departamento"))

df_divipola_limpio = df_divipola \
    .withColumn("nom_mpio_limpio", homologar_territorio("nom_mpio")) \
    .withColumn("dpto_divi_limpio", homologar_territorio("dpto"))

# Ejecutamos el Join
df_final = df_integrado_limpio.join(
    df_divipola_limpio,
    (df_integrado_limpio["ciudad_limpia"] == df_divipola_limpio["nom_mpio_limpio"]) & 
    (df_integrado_limpio["dpto_limpio"] == df_divipola_limpio["dpto_divi_limpio"]),
    how="left"
)

df_final = df_final.drop("ciudad_limpia", "dpto_limpio", "nom_mpio_limpio", "dpto_divi_limpio", "latitud", "longitud")


# ==========================================
# 7. Reporte de registros sin cruce territorial
# ==========================================
# Excluimos los "No Definido" y "Colombia" porque son imposibles de cruzar por naturaleza
df_sin_cruce = df_final.filter(
    F.col("cod_mpio").isNull() & 
    ~F.upper(F.col("ciudad")).isin("NO DEFINIDO", "COLOMBIA", "SIN DESCRIPCION") &
    ~F.upper(F.col("departamento")).isin("NO DEFINIDO", "COLOMBIA", "SIN DESCRIPCION")
)

print(f"Total de contratos SIN cruce territorial (Verdaderos Errores): {df_sin_cruce.count()}")

# Agrupamos los ofensores REALES para ver qué nos falta arreglar
df_ofensores = df_sin_cruce.groupBy("departamento", "ciudad") \
    .count() \
    .orderBy(F.col("count").desc())

display(df_ofensores)

## **Conexion segura a mongo mediante widget**

In [0]:
# ====================================================================================================
# 🔐 CONFIGURACIÓN DE CONEXIÓN SEGURA EN ENTORNO COMPARTIDO (ANTI-FILTRADO)
# ====================================================================================================
from pyspark.sql import functions as F
import os

# 🎭 1. Creamos un campo de entrada dinámico en la parte superior del notebook
dbutils.widgets.text("MONGO_CONNECTION_STRING", "", "Cadena de Conexión MongoDB Atlas")

# 📥 2. Capturamos en memoria lo que el usuario digite en esa casilla
mongo_uri_segura = dbutils.widgets.get("MONGO_CONNECTION_STRING")

# 🚨 3. Validación interactiva
if not mongo_uri_segura or mongo_uri_segura.strip() == "":
    print("⚠️  BLOQUEO PREVENTIVO DE SEGURIDAD:")
    print("👉 Por favor, ve a la parte superior de este notebook, busca el cuadro que dice")
    print("   'Cadena de Conexión MongoDB Atlas' y pega tu URL de Mongo con tu contraseña.")
    print("   (Ejemplo: mongodb+srv:...)")
    print("-" * 84)
    raise ValueError("Falta configurar la credencial en el Widget superior para continuar.")
else:
    print("✅ Credencial cargada con éxito en la memoria volátil de la sesión.")
    print("🔒 El código fuente está protegido. Puedes exportar con seguridad a GitHub.")

✅ Credencial cargada con éxito en la memoria volátil de la sesión.
🔒 El código fuente está protegido. Puedes exportar con seguridad a GitHub.


# 🚀 MIGRACIÓN INTEGRAL: CONEXIÓN ROBUSTA Y CARGA A MONGODB ATLAS


In [0]:
# ====================================================================================================
# 🚀 MIGRACIÓN INTEGRAL: CONEXIÓN ROBUSTA Y CARGA A MONGODB ATLAS (CON BATCHING)
# ====================================================================================================
import json
import os
import certifi
from pymongo import MongoClient

print("=" * 80)
print("🍃 INICIANDO CARGA MASIVA A MONGODB ATLAS (CAPA NOSQL)")
print("=" * 80)

# 🔒 CONEXIÓN SEGURA
try:
    mongo_uri_segura = dbutils.widgets.get("MONGO_CONNECTION_STRING")
except Exception:
    try:
        mongo_uri_segura = dbutils.widgets.get("MI_CONEXION_MONGO")
    except Exception:
        # Por si no tienes widgets, pega tu URI directamente aquí como texto
        mongo_uri_segura = "TU_URI_DE_ATLAS_AQUI" 

if not mongo_uri_segura or mongo_uri_segura.strip() == "":
    raise ValueError("❌ Error: La URI de conexión no está configurada.")

print("✅ Conexión lista.")

# --- CONEXIÓN AL CLIENTE DE MONGODB ATLAS ---
try:
    client = MongoClient(
        mongo_uri_segura,
        tls=True,
        tlsCAFile=certifi.where(),
        tlsAllowInvalidCertificates=True
    )
    db = client["taller_final_secop"]
    print("   ✅ Conexión establecida de forma segura con los Shards de Atlas.")
except Exception as e:
    print(f"   ❌ Error de conexión: {e}")
    raise

# --- CAPA DE RUTAS FISICAS ---
base_path = "/Volumes/workspace/default/taller_final/raw/mongodb_collections"

# Las carpetas de Spark no terminan en .json
colecciones_a_cargar = {
    "contratos_operativos": f"{base_path}/contratos_operativos",
    "alertas_revision": f"{base_path}/alertas_revision",
    "entidades_resumen": f"{base_path}/entidades_resumen",
    "proveedores_resumen": f"{base_path}/proveedores_resumen",
    "temas_resumen": f"{base_path}/temas_resumen",
    "metadata_pipeline": f"{base_path}/metadata_pipeline"
}

# --- INYECCIÓN DOCUMENTAL MASIVA EN BUCLE (CON SISTEMA DE LOTES/BATCH) ---
for nombre_coleccion, ruta_carpeta in colecciones_a_cargar.items():
    print(f"\n📦 Procesando colección: '{nombre_coleccion}'...")
    documentos_json = []
    
    try:
        if os.path.exists(ruta_carpeta):
            # Spark guarda la data en sub-archivos dentro de la carpeta
            for archivo in os.listdir(ruta_carpeta):
                if archivo.endswith(".json") and not archivo.startswith(".") and not archivo.startswith("_"):
                    ruta_completa = os.path.join(ruta_carpeta, archivo)
                    # Leemos cada archivo de partición
                    with open(ruta_completa, "r", encoding="utf-8") as f:
                        for linea in f:
                            if linea.strip():
                                documentos_json.append(json.loads(linea.strip()))
            
            if documentos_json:
                coleccion_mongo = db[nombre_coleccion]
                
                # Limpieza preventiva para no duplicar datos
                coleccion_mongo.delete_many({})
                
                # ========================================================
                # 🚀 SISTEMA DE INYECCIÓN POR LOTES (Para evitar Timeout)
                # ========================================================
                BATCH_SIZE = 10000  # Enviamos de a 10.000 documentos a la vez
                total_insertados = 0
                
                for i in range(0, len(documentos_json), BATCH_SIZE):
                    lote = documentos_json[i : i + BATCH_SIZE]
                    coleccion_mongo.insert_many(lote)
                    total_insertados += len(lote)
                    
                    # Imprimir el progreso solo para contratos_operativos para no llenar la pantalla
                    if nombre_coleccion == "contratos_operativos":
                        print(f"      -> {total_insertados:,} subidos...")
                        
                print(f"   📥 ¡Éxito total! Se subieron {total_insertados:,} documentos anidados a Atlas.")
            else:
                print(f"   ⚠️  La carpeta existe pero no contiene archivos .json válidos.")
        else:
            print(f"   ❌ No se encontró la carpeta física en: {ruta_carpeta}")
            
    except Exception as e:
        print(f"   ❌ Error insertando datos en {nombre_coleccion}: {e}")

print("\n" + "=" * 80)
print("🎉 ¡PROCESO CONCLUIDO! Las 6 colecciones ya están disponibles en MongoDB Atlas.")
print("=" * 80)

🍃 INICIANDO CARGA MASIVA A MONGODB ATLAS (CAPA NOSQL)
✅ Conexión lista.
   ✅ Conexión establecida de forma segura con los Shards de Atlas.

📦 Procesando colección: 'contratos_operativos'...
      -> 10,000 subidos...
      -> 20,000 subidos...
      -> 30,000 subidos...
      -> 40,000 subidos...
      -> 50,000 subidos...
      -> 60,000 subidos...
      -> 70,000 subidos...
      -> 80,000 subidos...
      -> 90,000 subidos...
      -> 100,000 subidos...
      -> 110,000 subidos...
      -> 120,000 subidos...
      -> 125,111 subidos...
   📥 ¡Éxito total! Se subieron 125,111 documentos anidados a Atlas.

📦 Procesando colección: 'alertas_revision'...
   📥 ¡Éxito total! Se subieron 125,111 documentos anidados a Atlas.

📦 Procesando colección: 'entidades_resumen'...
   📥 ¡Éxito total! Se subieron 3,340 documentos anidados a Atlas.

📦 Procesando colección: 'proveedores_resumen'...
   📥 ¡Éxito total! Se subieron 519,239 documentos anidados a Atlas.

📦 Procesando colección: 'temas_resumen'